# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


# Model Building

In [ ]:
# Create a master folder to keep all files created when executing the below code cells
import os
os.makedirs("tourism_project", exist_ok=True)

In [ ]:
# Create a folder for storing the model building files
os.makedirs("tourism_project/model_building", exist_ok=True)

## Data Registration

In [ ]:
os.makedirs("tourism_project/data", exist_ok=True)

Once the **data** folder created after executing the above cell, please upload the **tourism.csv** in to the folder

## Data Preparation

In [ ]:
from pathlib import Path
import pandas as pd
from src.pipeline import clean_data, load_source_csv, split_and_save

# Load directly from the Hugging Face dataset repository when no local file exists.
SOURCE_DATASET = "sprd12/Great_Learning"
raw = load_source_csv("data/tourism.csv")
cleaned = clean_data(raw)
train_df, test_df = split_and_save(cleaned, "data/processed")
print(f"Source shape: {raw.shape}; cleaned shape: {cleaned.shape}")
print(f"Train shape: {train_df.shape}; test shape: {test_df.shape}")
display(train_df.head())


## Model Training and Registration with Experimentation Tracking

In [ ]:
import json
from src.pipeline import train_and_evaluate

# The pipeline imputes missing values, one-hot encodes categoricals, and tunes a Random Forest.
metrics = train_and_evaluate(
    "data/processed/train.csv", "data/processed/test.csv", "artifacts"
)
print(json.dumps({k: v for k, v in metrics.items() if k != "classification_report"}, indent=2))
print("Upload artifacts/model.joblib to https://huggingface.co/sprd12/RandomForest")


# Deployment

## Dockerfile

In [ ]:
os.makedirs("tourism_project/deployment", exist_ok=True)

In [ ]:
%%writefile tourism_project/deployment/Dockerfile
# Use a minimal base image with Python 3.9 installed
FROM python:3.9

# Set the working directory inside the container to /app
WORKDIR /app

# Copy all files from the current directory on the host to the container's /app directory
COPY . .

# Install Python dependencies listed in requirements.txt
RUN pip3 install -r requirements.txt

RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user \
	PATH=/home/user/.local/bin:$PATH

WORKDIR $HOME/app

COPY --chown=user . $HOME/app

# Define the command to run the Streamlit app on port "8501" and make it accessible externally
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

Writing tourism_project/deployment/Dockerfile


## Streamlit App

Please ensure that the web app script is named `app.py`.

In [ ]:
%%writefile tourism_project/deployment/app.py
import os
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download

MODEL_REPO_ID = os.environ.get("HF_MODEL_REPO_ID", "sprd12/RandomForest")
@st.cache_resource
def load_model():
    path = Path("model.joblib")
    if not path.exists():
        path = Path(hf_hub_download(MODEL_REPO_ID, "model.joblib", repo_type="model"))
    return joblib.load(path)

st.title("Wellness Tourism Package Predictor")
with st.form("customer_form"):
    age = st.number_input("Age", 18, 100, 35)
    city_tier = st.selectbox("City tier", [1, 2, 3])
    income = st.number_input("Monthly income", 0, 200000, 30000)
    passport = st.selectbox("Passport", [0, 1])
    own_car = st.selectbox("Own car", [0, 1])
    submitted = st.form_submit_button("Predict")
if submitted:
    customer = pd.DataFrame([{"Age": age, "CityTier": city_tier, "MonthlyIncome": income, "Passport": passport, "OwnCar": own_car, "TypeofContact": "Self Inquiry", "Occupation": "Salaried", "Gender": "Female", "NumberOfPersonVisiting": 2, "PreferredPropertyStar": 3, "MaritalStatus": "Single", "NumberOfTrips": 2, "NumberOfChildrenVisiting": 0, "Designation": "Manager", "PitchSatisfactionScore": 3, "ProductPitched": "Deluxe", "NumberOfFollowups": 3, "DurationOfPitch": 15}])
    st.metric("Purchase probability", f"{load_model().predict_proba(customer)[0, 1]:.1%}")


## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

In [ ]:
%%writefile tourism_project/deployment/requirements.txt
pandas>=2.1,<3
scikit-learn>=1.3,<2
joblib>=1.3,<2
huggingface_hub>=0.20,<1
streamlit>=1.30,<2


# Hosting

In [ ]:
%%writefile tourism_project/deployment/deploy_space.py
import os
from pathlib import Path
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
space_id = os.environ["HF_SPACE_REPO_ID"]
api.create_repo(space_id, repo_type="space", space_sdk="docker", exist_ok=True)
for name in ["app.py", "Dockerfile", "requirements.txt"]:
    api.upload_file(name, Path(name).name, space_id, repo_type="space")
print(f"Deployed to https://huggingface.co/spaces/{space_id}")


# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.

```
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server starts
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-traning,data-prep,register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

In [ ]:
%%writefile tourism_project/deployment/deploy_space.py
import os
from pathlib import Path
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
space_id = os.environ["HF_SPACE_REPO_ID"]
api.create_repo(space_id, repo_type="space", space_sdk="docker", exist_ok=True)
for file_name in ["app.py", "Dockerfile", "requirements.txt"]:
    api.upload_file(file_name, Path(file_name).name, space_id, repo_type="space")
print(f"Deployed to https://huggingface.co/spaces/{space_id}")


## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [ ]:
# Install Git
!apt-get install git

# Set your Git identity (replace with your details)
!git config --global user.email "<-------GitHub Email Address------->"
!git config --global user.name "<--------GitHub UserName--------->"

# Clone your GitHub repository
!git clone https://github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Move your folder to the repository directory
!mv /content/tourism_project/ /content/<--------GitHub Reponame--------->

In [ ]:
# Change directory to the cloned repository
%cd <--------GitHub Reponame--------->/

# Add the new folder to Git
!git add .

# Commit the changes
!git commit -m "first commit"

# Push to GitHub (you'll need your GitHub credentials; use a personal access token if 2FA enabled)
!git push https://<--------GitHub UserName--------->:<--------GitHub Token--------->@github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

In [ ]:
from pathlib import Path
import json
print("Project files:")
for path in sorted(Path(".").glob("**/*")):
    if path.is_file() and ".git" not in path.parts:
        print(path)
if Path("artifacts/metrics.json").exists():
    print(json.loads(Path("artifacts/metrics.json").read_text()))


- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

In [ ]:
from IPython.display import Markdown, display
display(Markdown("""
Model repository: https://huggingface.co/sprd12/RandomForest
Public Space URL: add the URL printed by `scripts/deploy_space.py` after deployment.
Add screenshots of the GitHub Actions run and Streamlit Space in this section.
"""))


<font size=6 color="navyblue">Power Ahead!</font>
___